# 🌱 Plant or Wait — Soil Probe Simulator (Notebook edition)

One soil reading → one decision, delivered **three ways**, exactly as the physical ESP32 unit will do it:

| Output | Who it serves | In this notebook |
|---|---|---|
| 20×4 **LCD** | readers, technicians | green character display |
| **Lights + icons** (🌱 green / ✋ red / ⚠ amber) | everyone, even without reading | glowing LEDs |
| **Voice** (Twi / English) + chime | farmers who cannot read English | real sound from your speakers |

Plus the **signal path** (probe → RS485 → ESP32 → outputs), a live **device terminal** (Modbus frames, DFPlayer commands, PLX-DAQ line, Wi-Fi upload), and an **offline mode** that stores readings and syncs later.

**How to run:** `Run All` (Kernel ▸ Restart & Run All). Then press **▶ Play panel demo** for the guided run, or pick a field condition and press **TEST SOIL**.

> 🔊 Browsers only allow sound after you click something on the page — the buttons count. Turn your volume up before recording the demo.

> The limits are illustrative values for a maize demo. An agronomist should set the real ones.

In [17]:
# Cell 1 — install (only needed the first time; harmless to re-run)
# Works in Jupyter, JupyterLab, VS Code and Google Colab.
%pip install -q ipywidgets numpy
# Optional: only if you want English voice via Google TTS instead of the browser voice
# %pip install -q gTTS

# Google Colab needs this line to display widgets. Safe to ignore elsewhere.
try:
    from google.colab import output
    output.enable_custom_widget_manager()
except Exception:
    pass

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.0.1 -> 26.3
[notice] To update, run: python.exe -m pip install --upgrade pip


## 1 · The brain: rules that run *on the device*
This is the part that matters most physically: the Plant/Wait answer comes from simple rules on the ESP32, so it works with **no internet**. A dead probe produces **CHECK DEVICE**, never "Plant".

In [18]:
# Cell 2 — data, rules, decision engine
import numpy as np, io, wave, base64, json, html, time, os, re, itertools
from IPython.display import display, HTML, clear_output, Javascript
import ipywidgets as W

FIELDS = [  # k=key, l=label, u=unit, s=slider step, src=which sensor supplies it
 dict(k='moist',l='Soil moisture',    u=' %',     min=0,   max=90,   s=1,  src='probe', main=True),
 dict(k='soilT',l='Soil temperature', u=' °C',    min=10,  max=45,   s=1,  src='probe', main=True),
 dict(k='pH',   l='Soil pH',          u='',       min=3,   max=9,    s=.1, src='probe', main=True),
 dict(k='airT', l='Air temperature',  u=' °C',    min=15,  max=45,   s=1,  src='air',   main=True),
 dict(k='rh',   l='Air humidity',     u=' %',     min=15,  max=100,  s=1,  src='air',   main=True),
 dict(k='ec',   l='Conductivity (EC)',u=' µS/cm', min=0,   max=3000, s=10, src='probe', main=False),
 dict(k='n',    l='Nitrogen',         u=' mg/kg', min=0,   max=200,  s=1,  src='probe', main=False),
 dict(k='p',    l='Phosphorus',       u=' mg/kg', min=0,   max=200,  s=1,  src='probe', main=False),
 dict(k='k',    l='Potassium',        u=' mg/kg', min=0,   max=300,  s=1,  src='probe', main=False),
 dict(k='press',l='Air pressure',     u=' hPa',   min=960, max=1040, s=1,  src='air',   main=False),
]

SCENES = {
 'ready': dict(l='Ready to plant',  probe=True,  v=dict(moist=38,soilT=26,pH=6.3,airT=29,rh=78,ec=620,n=62,p=28,k=140,press=1008)),
 'dry':   dict(l='Too dry',         probe=True,  v=dict(moist=24,soilT=26,pH=6.3,airT=28,rh=82,ec=480,n=55,p=24,k=130,press=1010)),
 'wet':   dict(l='Waterlogged',     probe=True,  v=dict(moist=72,soilT=24,pH=6.2,airT=25,rh=94,ec=540,n=48,p=22,k=118,press=1002)),
 'acid':  dict(l='Acidic soil',     probe=True,  v=dict(moist=40,soilT=26,pH=4.8,airT=29,rh=75,ec=410,n=40,p=12,k=95, press=1009)),
 'heat':  dict(l='Hot, dry spell',  probe=True,  v=dict(moist=30,soilT=32,pH=6.5,airT=39,rh=34,ec=700,n=58,p=26,k=135,press=1012)),
 'fault': dict(l='Probe unplugged', probe=False, v=dict(moist=38,soilT=26,pH=6.3,airT=29,rh=78,ec=620,n=62,p=28,k=140,press=1008)),
}

# Voice clips = the numbered files on the DFPlayer SD card  (/01/ English, /02/ Twi)
CLIPS = {
 1: dict(en='You can plant today.',                      tw='Wubetumi adua nnɛ.'),
 2: dict(en='Wait. Do not plant today.',                 tw='Twɛn kakra. Nnua nnɛ.'),
 3: dict(en='Ask someone to check this device.',         tw='Ma obi nhwɛ afidie no.'),
 10:dict(en='The soil is in good condition.',            tw='Asase no yɛ papa.'),
 11:dict(en='The soil is too dry.',                      tw='Asase no awo dodo.'),
 12:dict(en='The soil holds too much water.',            tw='Nsuo a ɛwɔ asase no mu dɔɔso dodo.'),
 13:dict(en='The soil is too cold.',                     tw='Asase no adwo dodo.'),
 14:dict(en='The soil is too hot.',                      tw='Asase no ayɛ hyew dodo.'),
 15:dict(en='The soil needs treatment before you plant.',tw='Ɛsɛ sɛ wɔsiesie asase no ansa na woadua.'),
 16:dict(en='The weather is too hot and dry.',           tw='Ewiem ayɛ hyew na awo dodo.'),
 20:dict(en='Test again in two days.',                   tw='San bɛhwɛ nna abien akyi.'),
}

def band(x, lo, hi): return 'low' if x < lo else 'high' if x > hi else 'ok'

def checks_for(v, probe_ok=True):
    # name, reading, rule text, status code, which sensor
    rows = [
     ('Soil moisture',   f"{v['moist']} %",             '25–60 %',    band(v['moist'],25,60), 'probe'),
     ('Soil temperature',f"{v['soilT']} °C",            '18–35 °C',   band(v['soilT'],18,35), 'probe'),
     ('Soil pH',         f"{v['pH']:.1f}",              '5.5–7.5',    band(v['pH'],5.5,7.5),  'probe'),
     ('Conductivity',    f"{v['ec']} µS/cm",            'Below 2000', 'high' if v['ec']>2000 else 'ok', 'probe'),
     ('Air',             f"{v['airT']} °C, {v['rh']} %",'Not above 35 °C with below 40 %',
                         'high' if (v['airT']>35 and v['rh']<40) else 'ok', 'air'),
     ('N, P, K',         f"{v['n']}, {v['p']}, {v['k']}",'Logged only', 'log', 'probe'),
     ('Air pressure',    f"{v['press']} hPa",           'Logged only', 'log', 'air'),
    ]
    out = []
    for name, now, rule, code_, src in rows:
        if (not probe_ok) and src == 'probe':
            now, code_ = 'No data', 'nodata'
        out.append(dict(name=name, now=now, rule=rule, code=code_, src=src))
    return out

def decide(v, probe_ok=True):
    checks = checks_for(v, probe_ok)
    if not probe_ok:
        return dict(state='fault', word='CHECK DEVICE', lcd='Probe no reply',
                    why='The soil probe did not reply, so no advice is given.', clips=[3], checks=checks)
    # first failing rule wins; (condition, reason-clip, LCD text, explanation)
    rules = [
     (v['moist'] < 25,  11, 'Soil too dry',      'The soil is below the moisture limit.'),
     (v['moist'] > 60,  12, 'Soil too wet',      'The soil is above the moisture limit.'),
     (v['soilT'] < 18,  13, 'Soil too cold',     'The soil is below the temperature limit.'),
     (v['soilT'] > 35,  14, 'Soil too hot',      'The soil is above the temperature limit.'),
     (v['pH'] < 5.5,    15, 'Soil too acidic',   'Soil pH is below the limit.'),
     (v['pH'] > 7.5,    15, 'Soil too alkaline', 'Soil pH is above the limit.'),
     (v['ec'] > 2000,   15, 'Soil too salty',    'Conductivity is above the limit.'),
     (v['airT'] > 35 and v['rh'] < 40, 16, 'Too hot and dry', 'The air is too hot and dry.'),
    ]
    hit = next((r for r in rules if r[0]), None)
    if hit:
        return dict(state='wait', word='WAIT', lcd=hit[2], why=hit[3], clips=[2, hit[1], 20], checks=checks)
    return dict(state='plant', word='PLANT', lcd='Conditions good',
                why='All checked readings are within the limits.', clips=[1, 10], checks=checks)

# ── self-test: the rules must behave exactly like the web demo
expected = {'ready':'plant','dry':'wait','wet':'wait','acid':'wait','heat':'wait','fault':'fault'}
for key, sc in SCENES.items():
    assert decide(sc['v'], sc['probe'])['state'] == expected[key], key
assert decide(SCENES['dry']['v'])['clips']  == [2, 11, 20]
assert decide(SCENES['wet']['v'])['clips']  == [2, 12, 20]
assert decide(SCENES['acid']['v'])['clips'] == [2, 15, 20]
assert decide(SCENES['heat']['v'])['clips'] == [2, 16, 20]
print('✅ Rules loaded and self-tested for all 6 field conditions.')

✅ Rules loaded and self-tested for all 6 field conditions.


## 2 · Sound
* **Chime** – a rising two-note for Plant, falling two-note for Wait, flat double-note for Check device (synthesised here, same notes as the hardware buzzer cue).
* **Voice** – numbered clips, like the DFPlayer SD card. English uses your browser's voice. **Twi uses real recordings** you drop in `clips/tw/001.wav` … (or upload them in section 5). Without recordings, Twi shows as on-screen subtitles so nothing is faked.

In [19]:
# Cell 3 — sound engine
SR = 22050
ENGLISH_ENGINE = 'browser'     # 'browser' (speechSynthesis)  or  'gtts' (needs: pip install gTTS + internet)
TWI_FALLBACK   = 'subtitles'   # 'subtitles' (honest)  or  'browser' (browser voice reads Twi text; pronunciation will be poor)

audio_out = W.Output(layout=W.Layout(height='0px', overflow='hidden'))   # hidden sound player

def tone_samples(freqs, dur=.26, step=.2):
    total = int(SR * (step * (len(freqs) - 1) + dur))
    out = np.zeros(total)
    for i, f in enumerate(freqs):
        n = int(SR * dur); t = np.arange(n) / SR
        env = np.minimum(1, t / .03) * np.exp(-t * 7)
        a = int(i * step * SR)
        out[a:a+n] += .45 * np.sin(2*np.pi*f*t) * env
    return out

def wav_b64(samples):
    pcm = (np.clip(samples, -1, 1) * 32767).astype('<i2').tobytes()
    buf = io.BytesIO()
    with wave.open(buf, 'wb') as w:
        w.setnchannels(1); w.setsampwidth(2); w.setframerate(SR); w.writeframes(pcm)
    return base64.b64encode(buf.getvalue()).decode()

def _play_html(src, mime):
    uid = next(_uid)   # fresh element each time so the same sound can replay
    with audio_out:
        clear_output(wait=True)
        display(HTML(f'<audio id="a{uid}" autoplay src="data:{mime};base64,{src}"></audio>'))
_uid = itertools.count()

def play_samples(samples):
    _play_html(wav_b64(samples), 'audio/wav'); return len(samples) / SR

def js_speak(text, lang='en-GB', rate=.92):
    with audio_out:
        clear_output(wait=True)
        display(Javascript(
            "window.speechSynthesis.cancel();"
            f"var u=new SpeechSynthesisUtterance({json.dumps(text)});u.lang={json.dumps(lang)};u.rate={rate};"
            "window.speechSynthesis.speak(u);"))

def est_speech_seconds(text): return len(text) * .075 + .65

def speak_english(text):
    if ENGLISH_ENGINE == 'gtts':
        try:
            from gtts import gTTS
            buf = io.BytesIO(); gTTS(text, lang='en', tld='co.uk').write_to_fp(buf)
            _play_html(base64.b64encode(buf.getvalue()).decode(), 'audio/mpeg')
            return est_speech_seconds(text)
        except Exception as e:
            print('gTTS unavailable, using browser voice:', e)
    js_speak(text); return est_speech_seconds(text)

# ── Twi recordings (real voices)
REC = {}   # clip number -> float samples @ 22.05 kHz

def read_wav_bytes(raw):
    with wave.open(io.BytesIO(raw)) as w:
        ch, sw, sr, n = w.getnchannels(), w.getsampwidth(), w.getframerate(), w.getnframes()
        data = w.readframes(n)
    if sw == 2:   x = np.frombuffer(data, '<i2').astype(float) / 32768
    elif sw == 1: x = (np.frombuffer(data, 'u1').astype(float) - 128) / 128
    else: raise ValueError('Use 16-bit or 8-bit PCM WAV')
    if ch > 1: x = x.reshape(-1, ch).mean(axis=1)
    if sr != SR: x = np.interp(np.linspace(0, len(x)-1, int(len(x) * SR / sr)), np.arange(len(x)), x)
    return x

def load_clip_folder(folder='clips/tw'):
    found = 0
    for n in CLIPS:
        p = os.path.join(folder, f'{n:03d}.wav')
        if os.path.exists(p):
            REC[n] = read_wav_bytes(open(p, 'rb').read()); found += 1
    return found

print(f'Loaded {load_clip_folder()} Twi recording(s) from clips/tw/.  English engine: {ENGLISH_ENGINE}.')

Loaded 11 Twi recording(s) from clips/tw/.  English engine: browser.


## 3 · What you see: the device, the lights, the readout, the signal path

In [20]:
# Cell 4 — HTML renderers (device, verdict, table, voice list, signal path, terminal)
DEV_CSS = '''<style>
.dev{background:#2B2F2C;color:#E9EDE7;border-radius:18px;padding:16px;border:3px solid #4B514C;width:372px;font-family:Consolas,monospace;box-shadow:0 12px 26px -16px #0009}
.dh{display:flex;justify-content:space-between;font-size:12px;color:#AEB6AF;margin-bottom:10px}
.wd{display:inline-block;width:9px;height:9px;border-radius:50%;margin-right:6px;background:#35E070}
.wd.down{background:#FF4B42}
.lcdbox{background:#111;padding:7px;border-radius:6px;position:relative}
.lcd{background:#A8C24A;color:#1A2A08;font:25px/1.1 'Courier New',Consolas,monospace;padding:8px 9px;white-space:pre;border-radius:2px;min-height:112px;transition:filter .3s}
.lcd.blur{filter:blur(4px)}
.lcdnote{position:absolute;left:0;right:0;top:42%;text-align:center;color:#fff;font:600 13px sans-serif;text-shadow:0 1px 3px #000}
.lights{display:flex;justify-content:space-around;margin-top:16px}
.lt{text-align:center;font-size:12px;color:#AEB6AF}
.led{width:50px;height:50px;border-radius:50%;margin:0 auto 5px;background:#3A403B;border:2px solid #555;display:flex;align-items:center;justify-content:center;font-size:22px;filter:grayscale(1) brightness(.6)}
.led.g.on{background:#35E070;box-shadow:0 0 26px #35E070;filter:none}
.led.r.on{background:#FF4B42;box-shadow:0 0 26px #FF4B42;filter:none}
.led.a.on{background:#FFB92E;box-shadow:0 0 26px #FFB92E;filter:none}
.led.a.pulse{background:#FFB92E;animation:pl .7s infinite alternate;filter:none}
@keyframes pl{from{opacity:.25}to{opacity:1;box-shadow:0 0 22px #FFB92E}}
.spk{margin-top:14px;display:flex;align-items:center;gap:10px;font-size:12px;color:#AEB6AF}
.spk .cone{font-size:24px;opacity:.35}.spk.on .cone{opacity:1;animation:pl .35s infinite alternate}
</style>'''

def device_html(lines, led, wifi_txt, wifi_ok, blur, speaking):
    txt = html.escape('\n'.join(l[:20].ljust(20) for l in (lines + ['']*4)[:4]))
    g = 'on' if led == 'plant' else ''
    r = 'on' if led == 'wait' else ''
    a = 'on' if led == 'fault' else 'pulse' if led == 'busy' else ''
    note = '<div class="lcdnote">Screen not used by this farmer</div>' if blur else ''
    return (DEV_CSS + f'''<div class="dev">
<div class="dh"><span>ESP32 FIELD UNIT</span><span><i class="wd {'' if wifi_ok else 'down'}"></i>{html.escape(wifi_txt)}</span></div>
<div class="lcdbox"><div class="lcd {'blur' if blur else ''}">{txt}</div>{note}</div>
<div class="lights">
 <div class="lt"><div class="led g {g}">🌱</div>PLANT</div>
 <div class="lt"><div class="led r {r}">✋</div>WAIT</div>
 <div class="lt"><div class="led a {a}">⚠️</div>CHECK</div>
</div>
<div class="spk {'on' if speaking else ''}"><span class="cone">🔊</span><span>DFPlayer + speaker · REPEAT button replays</span></div>
</div>''')

def verdict_html(state=None, word='Waiting', why='Press TEST SOIL.'):
    pal = {'plant':('#DCEFE2','#14763A'), 'wait':('#F8E0DE','#BC2626'), 'fault':('#F6E9CF','#8E5C10'), None:('#E8ECE6','#33413A')}
    bg, fg = pal.get(state, pal[None])
    return (f'<div style="background:{bg};color:{fg};border-radius:10px;padding:12px 16px;width:372px;font-family:sans-serif">'
            f'<div style="font-size:30px;font-weight:800;letter-spacing:.02em">{html.escape(word)}</div>'
            f'<div style="font-size:14px;color:#2b2b2b;margin-top:2px">{html.escape(why)}</div></div>')

def hear_html(r, lang, active=-1):
    if lang == 'off':
        return '<div style="font:13px sans-serif;color:#667;width:372px">🔇 Voice is off. Choose Twi or English on the device.</div>'
    if not r: return ''
    items = ''
    for i, n in enumerate(r['clips']):
        on = (i == active)
        sty = 'background:#DFE5F6;border-left:4px solid #27408B;font-weight:700;' if on else 'border-left:4px solid #ccd;'
        main = CLIPS[n]['tw'] if lang == 'tw' else CLIPS[n]['en']
        sub = f"<div style='font-size:12px;color:#556;font-weight:400'>{html.escape(CLIPS[n]['en'])}</div>" if lang == 'tw' else ''
        items += f"<div style='{sty}padding:6px 10px;margin:4px 0;color:#18261F;border-radius:4px'>🔊 {html.escape(main)}{sub}</div>"
    return f"<div style='font:14px sans-serif;width:372px'><div style='font-size:12px;letter-spacing:.06em;color:#55645B;text-transform:uppercase'>What the farmer hears</div>{items}</div>"

PILL = {'ok':('#DCEFE2','#14763A','OK'), 'low':('#F8E0DE','#BC2626','Too low'), 'high':('#F8E0DE','#BC2626','Too high'),
        'log':('#E4E8EE','#44505E','Logged'), 'nodata':('#F6E9CF','#8E5C10','No data')}

def table_html(r):
    if not r: return '<div style="font:13px sans-serif;color:#667">Press TEST SOIL to compare each reading with its rule.</div>'
    rows = ''
    for c in r['checks']:
        bg, fg, t = PILL[c['code']]
        rows += (f"<tr><td style='padding:5px 8px'>{c['name']}</td><td style='padding:5px 8px;font-family:Consolas,monospace'>{html.escape(c['now'])}</td>"
                 f"<td style='padding:5px 8px;color:#55645B'>{html.escape(c['rule'])}</td>"
                 f"<td style='padding:5px 8px'><span style='background:{bg};color:{fg};padding:2px 9px;border-radius:99px;font-weight:700;font-size:12px'>{t}</span></td></tr>")
    return ("<table style='border-collapse:collapse;font:13.5px sans-serif;color:#18261F;background:#F8FAF4;border-radius:8px;overflow:hidden'>"
            "<tr style='background:#E1E8DC;text-align:left'><th style='padding:6px 8px'>Reading</th><th style='padding:6px 8px'>Now</th><th style='padding:6px 8px'>Rule</th><th style='padding:6px 8px'>Result</th></tr>"
            + rows + "</table>")

def path_html(active=(), down=()):
    def node(i, title, sub):
        bg, bd = ('#DFE5F6', '#27408B') if i in active else ('#F8E0DE', '#BC2626') if i in down else ('#F8FAF4', '#CBD4C8')
        return (f"<div style='background:{bg};border:2px solid {bd};border-radius:8px;padding:6px 10px;margin:4px 0;color:#18261F;min-width:150px'>"
                f"<b style='font-size:13px'>{title}</b><div style='font-size:11px;color:#55645B'>{sub}</div></div>")
    arrow = "<div style='align-self:center;font-size:22px;color:#55645B;padding:0 6px'>➜</div>"
    left  = node('probe','7-in-1 soil probe','moisture · temp · pH · EC · NPK') + node('max','MAX3485','Modbus RS485 → UART') + node('bme','BME280','air temp · humidity · pressure (I²C)')
    mid   = node('esp','ESP32','Plant / Wait rules · works with no internet')
    right = (node('lcd','20×4 LCD','readers') + node('lights','Lights + icons','every farmer') + node('voice','DFPlayer + speaker','Twi-speaking farmer')
             + node('usb','USB serial','PLX-DAQ → Excel') + node('wifi',"Wi-Fi","→ Frank's platform"))
    return (f"<div style='display:flex;flex-wrap:wrap;font-family:sans-serif'>"
            f"<div>{left}</div>{arrow}<div style='align-self:center'>{mid}</div>{arrow}<div>{right}</div></div>")

LOG = []
def hexs(a): return ' '.join(f'{b:02X}' for b in a)
def crc16(a):
    c = 0xFFFF
    for b in a:
        c ^= b
        for _ in range(8): c = (c >> 1) ^ 0xA001 if c & 1 else c >> 1
    return [c & 255, c >> 8]
def modbus_req(): f = [1,3,0,0,0,7]; return f + crc16(f)
def modbus_resp(v):
    regs = [round(v['moist']*10), round(v['soilT']*10), int(v['ec']), round(v['pH']*10), int(v['n']), int(v['p']), int(v['k'])]
    f = [1,3,14]
    for x in regs: f += [x >> 8, x & 255]
    return f + crc16(f)
def df_frame(folder, track):                      # DFPlayer Mini "play folder/track" command
    s = 0xFF + 6 + 0x0F + 0 + folder + track; cs = (0x10000 - s) & 0xFFFF
    return [0x7E, 0xFF, 6, 0x0F, 0, folder, track, cs >> 8, cs & 255, 0xEF]

def term_html():
    body = ''.join(f"<div><span style='color:#9DB8F5'>[{t}]</span> {m}</div>" for t, m in LOG[-40:])
    return ("<div style='background:#101512;color:#C8D8CC;font:12px/1.5 Consolas,monospace;padding:10px 12px;border-radius:8px;"
            "height:210px;overflow-y:auto;white-space:pre-wrap'>" + (body or "<span style='color:#7F9587'>Device terminal. Nothing sent yet.</span>") + "</div>")
print('Renderers ready.')

Renderers ready.


## 4 · The simulator
Left: choose the **farmer**, the **field condition**, tweak sensor values, break the Wi-Fi or unplug the probe.  
Middle: the **physical unit**, **TEST SOIL** and **REPEAT**.  Right: each reading checked against its rule.  
Below: the **signal path** lighting up stage by stage and the **device terminal**.

In [21]:
# Cell 5 — simulator state, widgets, logic, layout
S = dict(v=dict(SCENES['dry']['v']), scene='dry', cust='reader', voice='off', wifi=True, probe=True,
         queue=0, res=None, seq=1, lcd=['Conditions changed.','Press TEST SOIL.','',''], led=None, lock=False)

# ---- widgets
dev_w     = W.HTML(); verdict_w = W.HTML(); hear_w = W.HTML(); table_w = W.HTML()
path_w    = W.HTML(); term_w = W.HTML(); narr_w = W.HTML()

cust_rb  = W.RadioButtons(options=[('Reads English','reader'),('Does not read English','nonreader'),('Technician','tech')], value='reader', layout=W.Layout(width='320px'))
scene_rb = W.RadioButtons(options=[(s['l'], k) for k, s in SCENES.items()], value='dry', layout=W.Layout(width='320px'))
voice_tb = W.ToggleButtons(options=[('Voice off','off'),('Twi','tw'),('English','en')], value='off', layout=W.Layout(width='372px'),
                           style=W.ToggleButtonsStyle(button_width='120px'))
wifi_cb  = W.Checkbox(value=True, description='Internet available', indent=False)
probe_cb = W.Checkbox(value=True, description='Soil probe connected', indent=False)
test_btn = W.Button(description='TEST SOIL', button_style='primary', icon='play', layout=W.Layout(width='180px', height='42px'))
rep_btn  = W.Button(description='REPEAT', icon='volume-up', layout=W.Layout(width='180px', height='42px'))
demo_btn = W.Button(description='▶ Play panel demo', button_style='success', layout=W.Layout(width='200px', height='40px'))

SL = {}
for f in FIELDS:
    cls = W.FloatSlider if f['s'] < 1 else W.IntSlider
    SL[f['k']] = cls(value=S['v'][f['k']], min=f['min'], max=f['max'], step=f['s'], description=f['l'], readout_format='.1f' if f['s'] < 1 else 'd',
                     continuous_update=False, layout=W.Layout(width='330px'), style={'description_width':'125px'})

# ---- helpers that repaint things
def wifi_label():
    return ('Wi-Fi OK', True) if S['wifi'] else ((f"{S['queue']} stored", False) if S['queue'] else ('Offline', False))

def paint_device(speaking=False):
    t, ok = wifi_label()
    dev_w.value = device_html(S['lcd'], S['led'], t, ok, S['cust'] == 'nonreader', speaking)

def set_lcd(lines, led=None):
    S['lcd'], S['led'] = lines, led; paint_device()

def paint_path(active=(), down=()): path_w.value = path_html(active, down)
def log(tag, msg): LOG.append((tag, msg)); term_w.value = term_html()

def lcd_result(r):
    v = S['v']; net = 'WiFi OK' if S['wifi'] else 'Offline'
    air = f"A:{v['airT']}C H:{v['rh']}% {net}"
    if r['state'] == 'fault': return ['*** CHECK DEVICE ***', r['lcd'], 'Check probe cable', air]
    return ['>> ' + r['word'], r['lcd'], f"M:{v['moist']}% T:{v['soilT']}C pH:{v['pH']:.1f}", air]

def mark_stale():
    S['res'] = None
    set_lcd(['Conditions changed.', 'Press TEST SOIL.', '', ''], None)
    verdict_w.value = verdict_html(); hear_w.value = hear_html(None, S['voice']); table_w.value = table_html(None)
    paint_path()

# ---- sound + voice sequence (chime, then numbered clips like the DFPlayer)
def speak(r, pause=time.sleep):
    chime = tone_samples({'plant':[523,784], 'wait':[587,392], 'fault':[440,440]}[r['state']])
    d = play_samples(chime); pause(d + .15)
    lang = S['voice']
    hear_w.value = hear_html(r, lang)
    if lang == 'off': return
    folder = 2 if lang == 'tw' else 1
    paint_device(speaking=True)
    for i, n in enumerate(r['clips']):
        hear_w.value = hear_html(r, lang, i)
        log('DFP', f"playFolder({folder},{n})  {hexs(df_frame(folder, n))}  → /{folder:02d}/{n:03d}.mp3")
        if lang == 'tw' and n in REC:
            dur = play_samples(REC[n])
        elif lang == 'en':
            dur = speak_english(CLIPS[n]['en'])
        else:                                   # Twi without a recording
            if TWI_FALLBACK == 'browser': js_speak(CLIPS[n]['tw'], 'en-GB')
            dur = est_speech_seconds(CLIPS[n]['tw'])
        pause(dur + .18)
    hear_w.value = hear_html(r, lang); paint_device(speaking=False)

# ---- the TEST SOIL sequence
def run_test(pause=time.sleep):
    v, probe = dict(S['v']), S['probe']
    for b in (test_btn, rep_btn, demo_btn): b.disabled = True
    try:
        set_lcd(['Testing soil...', '', '', ''], 'busy')
        verdict_w.value = verdict_html(None, 'Reading', 'Collecting data from the sensors.'); hear_w.value = ''
        log('RS485', f"#{S['seq']} TX {hexs(modbus_req())}"); S['seq'] += 1
        paint_path(active={'probe','max','esp'}); pause(.65)
        if not probe:
            log('RS485', 'no reply (timeout, 3 retries)'); paint_path(active={'esp'}, down={'probe','max'}); pause(.5)
        else:
            log('RS485', f"RX {hexs(modbus_resp(v))}")
            log('I2C', f"BME280  air {v['airT']} °C · RH {v['rh']} % · {v['press']} hPa")
            paint_path(active={'probe','max','bme','esp'}); pause(.6)
        r = decide(v, probe); S['res'] = r
        log('ESP32', f"rules → {r['word']}  ({r['lcd']})")
        set_lcd(lcd_result(r), r['state'])
        verdict_w.value = verdict_html(r['state'], r['word'], r['why']); table_w.value = table_html(r)
        outs = {'esp','lcd','lights'} | ({'voice'} if S['voice'] != 'off' else set()) | {'usb'} | ({'wifi'} if S['wifi'] else set())
        paint_path(active=outs | ({'probe','max','bme'} if probe else {'bme'}), down=set() if probe else {'probe','max'})
        if probe:
            log('USB', f"DATA,TIME,{v['moist']},{v['soilT']},{v['pH']:.1f},{v['ec']},{v['n']},{v['p']},{v['k']},{v['airT']},{v['rh']},{v['press']},{r['word']}")
        else:
            log('USB', f"DATA,TIME,NA,NA,NA,NA,NA,NA,NA,{v['airT']},{v['rh']},{v['press']},CHECK_DEVICE")
        if S['wifi']: log('WiFi', "POST /api/readings → 200 OK  (sent to Frank's platform)")
        else:
            S['queue'] += 1; paint_device()
            log('WiFi', f"offline · reading stored on device (queue = {S['queue']})")
        pause(.4)
        speak(r, pause)
    finally:
        for b in (test_btn, rep_btn, demo_btn): b.disabled = False

# ---- widget events
def on_scene(c):
    if S['lock']: return
    sc = SCENES[c['new']]; S['scene'] = c['new']; S['lock'] = True
    for k, val in sc['v'].items(): SL[k].value = val; S['v'][k] = val
    probe_cb.value = sc['probe']; S['probe'] = sc['probe']; S['lock'] = False
    mark_stale()
def on_slider(k):
    def h(c):
        if S['lock']: return
        S['v'][k] = c['new']; mark_stale()
    return h
def on_cust(c):
    S['cust'] = c['new']; S['lock'] = True
    voice_tb.value = {'reader':'off', 'nonreader':'tw', 'tech':'off'}[c['new']]; S['voice'] = voice_tb.value; S['lock'] = False
    hear_w.value = hear_html(S['res'], S['voice']); paint_device()
def on_voice(c):
    if S['lock']: return
    S['voice'] = c['new']; hear_w.value = hear_html(S['res'], S['voice'])
def on_wifi(c):
    S['wifi'] = c['new']
    if S['wifi'] and S['queue']:
        log('WiFi', f"back online → sending {S['queue']} stored reading(s) to Frank's platform → 200 OK"); S['queue'] = 0
    elif not S['wifi']: log('WiFi', 'connection lost · device keeps working offline')
    paint_device()
    if S['res']: S['lcd'] = lcd_result(S['res']); paint_device()
def on_probe(c):
    if S['lock']: return
    S['probe'] = c['new']; mark_stale()
def on_repeat(_):
    if S['res'] is None: log('DEV', 'REPEAT pressed but there is no result yet — press TEST SOIL first'); return
    for b in (test_btn, rep_btn, demo_btn): b.disabled = True
    try: speak(S['res'])
    finally:
        for b in (test_btn, rep_btn, demo_btn): b.disabled = False

scene_rb.observe(on_scene, 'value'); cust_rb.observe(on_cust, 'value'); voice_tb.observe(on_voice, 'value')
wifi_cb.observe(on_wifi, 'value'); probe_cb.observe(on_probe, 'value')
for k, s_ in SL.items(): s_.observe(on_slider(k), 'value')
test_btn.on_click(lambda _: run_test()); rep_btn.on_click(on_repeat)

# ---- guided panel demo (what you record for the video)
def narrate(text): narr_w.value = ("<div style='background:#DFE5F6;color:#18261F;border-radius:8px;padding:10px 16px;font:17px sans-serif;font-weight:600'>🎙️ " + html.escape(text) + "</div>")

def play_demo(_=None):
    S['lock'] = False
    def setup(scene, cust, voice, wifi=True):
        wifi_cb.value = wifi
        scene_rb.value = scene; cust_rb.value = cust; voice_tb.value = voice
    steps = [
     ('Field is ready. A farmer who reads English uses the screen: PLANT, with a rising chime.', 'ready','reader','off',True),
     ('Soil is too dry. The red light and the screen say WAIT, with the reason.',                'dry','reader','off',True),
     ('Waterlogged soil: WAIT again, but the reason changes. The rules, not guesses.',           'wet','reader','off',True),
     ('Acidic soil, and the farmer cannot read English: the lights and the Twi voice carry the message.', 'acid','nonreader','tw',True),
     ('Hot, dry spell: air conditions alone can block planting.',                                 'heat','reader','en',True),
     ('Probe cable cut: amber CHECK DEVICE. A broken sensor must never say PLANT.',               'fault','nonreader','tw',True),
     ('Internet is down. The device still answers, and stores the reading to send later.',       'ready','reader','off',False),
    ]
    for text, sc, cu, vo, wf in steps:
        narrate(text); setup(sc, cu, vo, wf); time.sleep(1.6); run_test(); time.sleep(1.4)
    narrate('Wi-Fi returns: the stored readings are sent to the platform. Same device, three outputs, no internet needed for the answer.')
    wifi_cb.value = True; time.sleep(2.5); narrate('Demo finished. Try your own field conditions with the controls on the left.')
demo_btn.on_click(play_demo)

# ---- layout
H = lambda s: W.HTML(f"<div style='font:600 12px sans-serif;letter-spacing:.06em;text-transform:uppercase;color:#55645B;margin:10px 0 2px'>{s}</div>")
more = W.Accordion(children=[W.VBox([SL[f['k']] for f in FIELDS if not f['main']])], layout=W.Layout(width='340px')); more.set_title(0, 'More readings')
left  = W.VBox([H('Who is using it'), cust_rb, H('Field condition'), scene_rb, H('Sensor values')] + [SL[f['k']] for f in FIELDS if f['main']]
               + [more, H('Connections'), wifi_cb, probe_cb], layout=W.Layout(width='350px'))
mid   = W.VBox([dev_w, W.HBox([test_btn, rep_btn]), H('Voice'), voice_tb, H('Decision'), verdict_w, hear_w], layout=W.Layout(width='390px'))
right = W.VBox([H('Readings against the rules'), table_w], layout=W.Layout(min_width='430px', flex='1 1 430px'))
top   = W.HBox([left, mid, right], layout=W.Layout(flex_flow='row wrap', align_items='flex-start', gap='14px'))
header = W.VBox([W.HTML("<h2 style='margin:0;font-family:sans-serif'>Plant or Wait · one soil reading, one decision, three ways</h2>"),
                 W.HBox([demo_btn, W.Box([narr_w], layout=W.Layout(flex='1'))], layout=W.Layout(align_items='center', gap='12px'))])
app = W.VBox([header, audio_out, top, H('Signal path: sensing stays the same, only the last step changes'), path_w, H('Live traffic on the device'), term_w])

# ---- first paint
S['lock'] = False; mark_stale(); paint_path(); term_w.value = term_html()
narrate('Pick a field condition and press TEST SOIL, or press Play panel demo.')
display(app)

In [22]:
# Cell 5b - load numbered Twi recordings
REC.clear()
NUMBERED_TWI_LOADED = load_clip_folder('clips/tw')
print(f'Loaded {NUMBERED_TWI_LOADED} numbered Twi recording(s) from clips/tw/.')
if NUMBERED_TWI_LOADED == len(CLIPS):
    print('Twi playback is clip-specific: each result uses only its required numbered recording.')


Loaded 11 numbered Twi recording(s) from clips/tw/.
Twi playback is clip-specific: each result uses only its required numbered recording.


## 5 · Twi voice recordings (optional, but best for the demo)
Without recordings the **Twi mode shows subtitles and the chime only** (we don't fake a Twi voice). For the real thing:

1. Have a Twi speaker record the 11 short lines below (phone voice recorder is fine), one file per line.
2. Export as **WAV** and name them by clip number: `001.wav`, `002.wav`, `003.wav`, `010.wav` … `020.wav`.
3. Either put them in a folder `clips/tw/` next to this notebook and re-run Cell 3, **or** upload them with the button below (works in Colab too).

On the real device the same numbers are the files on the DFPlayer SD card: `/02/001.mp3` … so recording once serves both the demo and the hardware. Let a second native speaker check every word.

In [23]:
# Cell 6 — upload Twi recordings (named 001.wav, 002.wav, 003.wav, 010.wav ... 020.wav)
status_w = W.HTML()
def show_status():
    rows = ''.join(f"<tr><td style='padding:3px 10px'>{n:03d}</td><td style='padding:3px 10px'>{html.escape(c['tw'])}</td>"
                   f"<td style='padding:3px 10px'>{'✅ loaded' if n in REC else '— subtitles only'}</td></tr>" for n, c in CLIPS.items())
    status_w.value = f"<table style='font:13px sans-serif;border-collapse:collapse'>{rows}</table>"
up = W.FileUpload(accept='.wav', multiple=True, description='Upload .wav clips')
def on_upload(c):
    items = up.value.values() if isinstance(up.value, dict) else up.value
    for it in items:
        name = it['name'] if isinstance(it, dict) else it.name
        content = it['content'] if isinstance(it, dict) else it.content
        m = re.match(r'(\d{1,3})', name)
        if m and int(m.group(1)) in CLIPS:
            try: REC[int(m.group(1))] = read_wav_bytes(bytes(content))
            except Exception as e: print('Could not read', name, e)
    show_status()
up.observe(on_upload, 'value'); show_status(); display(W.VBox([up, status_w]))

## 6 · From notebook to hardware (connect the dots)

| In this notebook | On the physical unit |
|---|---|
| `decide(v, probe_ok)` | the rules in the ESP32 firmware: same limits, same first-failure-wins order |
| `modbus_req()` / `modbus_resp()` | Modbus RTU read of 7 registers from the soil probe through the MAX3485 (CRC16 is real) |
| BME280 line | air temperature, humidity, pressure over I²C |
| `lcd_result()` | 20×4 I²C LCD, 20 characters per line |
| 🌱 ✋ ⚠️ lights | three LEDs, each with a printed icon so colour-blind users are covered |
| `df_frame(folder, track)` | the real DFPlayer Mini serial command, `/01/` English, `/02/` Twi |
| `DATA,TIME,…` line | what PLX-DAQ receives over USB serial to fill Excel |
| Wi-Fi queue | readings kept in ESP32 flash while offline, uploaded to the platform later |
| `CHECK DEVICE` state | probe timeout ⇒ amber light and voice message, never "Plant" |

**Next steps on the real build:** have an agronomist set the limits (they are placeholders for maize), record and check the Twi clips with native speakers, then test the unit with farmers in sunlight and noise before relying on it.